<a href="https://colab.research.google.com/github/mf2056/F20AA_CW2/blob/main/Step4%265.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

In [3]:
train = pd.read_csv("/content/drive/MyDrive/train_processed.csv")

train.head()

,text,rating,tokens,stemmed,lemmatized,text_stem,text_lemma,review_length
0,this place is terrible the people in charge ar...,2,"['place', 'terrible', 'people', 'charge', 'wor...","['place', 'terribl', 'peopl', 'charg', 'worst'...","['place', 'terrible', 'people', 'charge', 'wor...",place terribl peopl charg worst part far yeah ...,place terrible people charge worst part far ye...,97
1,terrible service and they are saying that i ne...,1,"['terrible', 'service', 'saying', 'never', 'us...","['terribl', 'servic', 'say', 'never', 'use', '...","['terrible', 'service', 'saying', 'never', 'us...",terribl servic say never use servic lie call n...,terrible service saying never used service lie...,48
2,absolutely terrible company they sent me to co...,1,"['absolutely', 'terrible', 'company', 'sent', ...","['absolut', 'terribl', 'compani', 'sent', 'col...","['absolutely', 'terrible', 'company', 'sent', ...",absolut terribl compani sent collect without a...,absolutely terrible company sent collection wi...,211
3,to find it either park in front of the tuesday...,4,"['find', 'either', 'park', 'front', 'tuesday',...","['find', 'either', 'park', 'front', 'tuesday',...","['find', 'either', 'park', 'front', 'tuesday',...",find either park front tuesday morn mall entra...,find either park front tuesday morning mall en...,66
4,mall location used their services for sedan ni...,4,"['mall', 'location', 'used', 'services', 'seda...","['mall', 'locat', 'use', 'servic', 'sedan', 'n...","['mall', 'location', 'used', 'service', 'sedan...",mall locat use servic sedan nice perhap inform...,mall location used service sedan nice perhaps ...,28


In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import classification_report

In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

In [13]:
train["text_lemma"] = train["text_lemma"].fillna("")
train["text_lemma"] = train["text_lemma"].astype(str)

train["text_stem"] = train["text_stem"].fillna("")
train["text_stem"] = train["text_stem"].astype(str)

X = train["text_lemma"]
Y = train["rating"]

In [8]:
train.shape

(288000, 8)

In [14]:
train.isnull().sum()

,0
text,43
rating,0
tokens,0
stemmed,0
lemmatized,0
text_stem,0
text_lemma,0
review_length,0


In [15]:
import pandas as pd
from sklearn.model_selection import GridSearchCV

def run_gridsearch(name, pipeline, params):
    # 1. Define multiple metrics
    scoring = {
        'Accuracy': 'accuracy',
        'MacroF1': 'f1_macro'
    }

    grid = GridSearchCV(
        pipeline,
        params,
        cv=3,
        n_jobs=-1,
        verbose=1,
        scoring=scoring,
        refit='MacroF1', # This tells it to pick the "Best Model" based on F1
        return_train_score=False
    )

    grid.fit(X, Y)

    # 2. Extract detailed results into a table
    results_df = pd.DataFrame(grid.cv_results_)

    # Select columns for the 3 folds and the means
    columns_to_show = ['params']
    # Add Accuracy columns
    columns_to_show += ['split0_test_Accuracy', 'split1_test_Accuracy', 'split2_test_Accuracy', 'mean_test_Accuracy']
    # Add F1 columns
    columns_to_show += ['split0_test_MacroF1', 'split1_test_MacroF1', 'split2_test_MacroF1', 'mean_test_MacroF1']

    report_table = results_df[columns_to_show].sort_values('mean_test_MacroF1', ascending=False)

    print(f"\n==== {name} DETAILED FOLD RESULTS ====")
    print(report_table.to_string(index=False))

    # 3. Print the absolute bests
    best_f1_idx = results_df['mean_test_MacroF1'].idxmax()
    best_acc_idx = results_df['mean_test_Accuracy'].idxmax()

    print(f"\n--- {name} Summary ---")
    print(f"Best Params (for F1): {grid.best_params_}")
    print(f"Best Mean Macro-F1: {results_df.loc[best_f1_idx, 'mean_test_MacroF1']:.4f}")
    print(f"Best Mean Accuracy: {results_df.loc[best_acc_idx, 'mean_test_Accuracy']:.4f}")

    return grid

In [16]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        max_features=50000
    )),
    ("clf", MultinomialNB())
])

nb_params = {
    "clf__alpha": [0.1, 0.5, 1.0]
}

nb_grid = run_gridsearch(
    "Naive Bayes",
    nb_pipeline,
    nb_params
)

Fitting 3 folds for each of 3 candidates, totalling 9 fits

==== Naive Bayes DETAILED FOLD RESULTS ====
             params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__alpha': 0.1}              0.637219              0.636490              0.635531            0.636413             0.465647             0.465948             0.465692           0.465763
{'clf__alpha': 0.5}              0.636573              0.636625              0.635854            0.636351             0.439673             0.441473             0.441371           0.440839
{'clf__alpha': 1.0}              0.632010              0.629823              0.629906            0.630580             0.405300             0.404131             0.404143           0.404525

--- Naive Bayes Summary ---
Best Params (for F1): {'clf__alpha': 0.1}
Best Mean Macro-F1: 0.4658
Best Mean Accuracy: 0.6364


In [17]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        max_features=50000
    )),
    ("clf", LogisticRegression(max_iter=1000))
])

lr_params = {
    "clf__C": [0.1, 1, 5]
}

lr_grid = run_gridsearch(
    "Logistic Regression",
    lr_pipeline,
    lr_params
)

Fitting 3 folds for each of 3 candidates, totalling 9 fits

==== Logistic Regression DETAILED FOLD RESULTS ====
         params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
  {'clf__C': 5}              0.638833               0.63749              0.638917            0.638413             0.507478             0.508243             0.511457           0.509059
  {'clf__C': 1}              0.657469               0.65549              0.654615            0.655858             0.503323             0.501926             0.502848           0.502699
{'clf__C': 0.1}              0.643781               0.64075              0.642073            0.642201             0.432389             0.428274             0.434776           0.431813

--- Logistic Regression Summary ---
Best Params (for F1): {'clf__C': 5}
Best Mean Macro-F1: 0.5091
Best Mean Accuracy: 0.6559


In [18]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        max_features=50000
    )),
    ("clf", LinearSVC())
])

svm_params = {
    "clf__C": [0.5, 1, 2]
}

svm_grid = run_gridsearch(
    "SVM",
    svm_pipeline,
    svm_params
)

Fitting 3 folds for each of 3 candidates, totalling 9 fits

==== SVM DETAILED FOLD RESULTS ====
         params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
{'clf__C': 0.5}              0.644823              0.643312              0.643302            0.643813             0.489808             0.489903             0.493333           0.491014
  {'clf__C': 1}              0.632250              0.631396              0.631490            0.631712             0.489493             0.490090             0.492892           0.490825
  {'clf__C': 2}              0.616948              0.615635              0.617083            0.616556             0.485217             0.485928             0.488676           0.486607

--- SVM Summary ---
Best Params (for F1): {'clf__C': 0.5}
Best Mean Macro-F1: 0.4910
Best Mean Accuracy: 0.6438


In [22]:
from sklearn.linear_model import SGDClassifier

sgd_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()), # Parameters handled by grid
    ("clf", SGDClassifier(max_iter=1000, tol=1e-3, random_state=25))
])

sgd_params = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],   # Step 3 requirement [cite: 9, 39]
    "tfidf__max_features": [10000, 50000],    # Step 3 requirement [cite: 38]
    "clf__alpha": [0.0001, 0.001, 0.01],     # Regularization strength
    "clf__loss": ['hinge', 'log_loss']        # hinge = SVM, log_loss = Logistic Regression
}

sgd_grid = run_gridsearch("SGD Classifier", sgd_pipeline, sgd_params)

Fitting 3 folds for each of 24 candidates, totalling 72 fits

==== SGD Classifier DETAILED FOLD RESULTS ====
                                                                                                     params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
   {'clf__alpha': 0.0001, 'clf__loss': 'hinge', 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.635687              0.633729              0.633042            0.634153             0.403259             0.402127             0.403054           0.402813
   {'clf__alpha': 0.0001, 'clf__loss': 'hinge', 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}              0.633646              0.630354              0.630990            0.631663             0.399573             0.397721             0.399907           0.399067
   {'clf__alpha': 0.0001, 'clf__loss': 'hinge', 'tfidf__max_fea

In [23]:
from sklearn.linear_model import RidgeClassifier

ridge_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", RidgeClassifier(random_state=28))
])

ridge_params = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],  # Comparison requirement [cite: 39]
    "tfidf__max_features": [10000, 50000],   # Representation requirement [cite: 38]
    "clf__alpha": [0.1, 1.0, 10.0]           # The "knob" for regularization
}

ridge_grid = run_gridsearch("Ridge Classifier", ridge_pipeline, ridge_params)

Fitting 3 folds for each of 12 candidates, totalling 36 fits

==== Ridge Classifier DETAILED FOLD RESULTS ====
                                                                          params  split0_test_Accuracy  split1_test_Accuracy  split2_test_Accuracy  mean_test_Accuracy  split0_test_MacroF1  split1_test_MacroF1  split2_test_MacroF1  mean_test_MacroF1
 {'clf__alpha': 1.0, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.642354              0.640563              0.640708            0.641208             0.476540             0.475902             0.479258           0.477233
 {'clf__alpha': 0.1, 'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 2)}              0.609510              0.607906              0.608927            0.608781             0.469053             0.468478             0.473190           0.470240
 {'clf__alpha': 0.1, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 2)}              0.645167              0.643719              0.64

In [ ]:
xgb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        max_features=30000
    )),
    ("clf", XGBClassifier(
        objective="multi:softmax",
        num_class=5,
        eval_metric="mlogloss",
        use_label_encoder=False
    ))
])

xgb_params = {
    "clf__n_estimators": [50, 100],
    "clf__max_depth": [4, 6]
}

xgb_grid = run_gridsearch(
    "XGBoost",
    xgb_pipeline,
    xgb_params
)